# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1 — “AI-Generated Content Is Penalized.”

The paper reports that, within its mostly AI-authored portfolio, age-controlled model cohorts do not show a simple blanket penalty tied only to AI use. The comparison covers 341,018 pieces of content across five AI models. The paper interprets the differences as more consistent with model choice, editing standards, process quality, and topic fit than with a simple AI-versus-human split.

**My methodology question:** Where does the outcome used to compare these cohorts come from, and is it measured after publication rather than being part of the information used to form the cohorts? It would also be useful to clarify how the age-controlled comparisons support the broader descriptive conclusion about AI penalties. Clearer outcome definitions and time alignment would make this finding easier to interpret.

### Finding 2 — “What Predicts Growth?”

The paper reports 71% holdout accuracy from logistic regression separating growing from declining pages. Content age is described as the strongest negative signal, while days visible and recent impressions are among the strongest positive signals. The paper describes these as descriptive indicators from the sampled active-content set.

**My methodology question:** Where does the growing-versus-declining label come from, and are the features measured before the period used to define that label? The paper reports an 80/20 holdout split, so it would also be useful to know whether related content or clients can appear in both training and holdout data. A grouped-by-client or time-aware validation could provide additional evidence about whether the measured accuracy carries over to unseen clients or future observations.

In [19]:
paper_claims_check = {
    "ai_content_pieces": 341018,
    "ai_model_families_compared": 5,
    "growth_model_holdout_accuracy_pct": 71,
    "growth_model_split": "80/20 holdout",
    "ml_pipeline_content_pieces": 61800,
}

print("AI-content pieces:", paper_claims_check["ai_content_pieces"])
print("AI model families compared:", paper_claims_check["ai_model_families_compared"])
print(
    "Growth model reported holdout accuracy:",
    f"{paper_claims_check['growth_model_holdout_accuracy_pct']}%"
)
print("Growth model validation:", paper_claims_check["growth_model_split"])
print(
    "ML pipeline size:",
    f"{paper_claims_check['ml_pipeline_content_pieces'] / 1000:.1f}K content pieces"
)

AI-content pieces: 341018
AI model families compared: 5
Growth model reported holdout accuracy: 71%
Growth model validation: 80/20 holdout
ML pipeline size: 61.8K content pieces


## 2. My model under an honest split (before/after)

My Week-5 model used a client-grouped train/test split, with 42 clients in training and 11 unseen clients in the test set. This prevented the same client from appearing in both sides of that evaluation.

For this validation audit, I re-ran the same modeling setup using 5-fold grouped validation by client_hash_id. Each fold holds out a different group of clients, so the model is evaluated on clients that were not used for training in that fold.

I kept the same target definition, historical features, Logistic Regression model, and Precision@K ranking metrics. The Week-5 single grouped holdout is shown alongside the Week-6 5-fold grouped mean to show how the measured results vary under a repeated grouped validation design.

The Week-6 results are not treated as proof that the model improved. The two numbers come from different validation designs. The results are evidence about the evaluated clients and data, and should be treated as directional decision-support rather than a guarantee of future traffic recovery or refresh success.

### 2A — Rebuild Week-5 modeling dataset

In [20]:
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42
con = duckdb.connect()

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con.execute("DROP SECRET IF EXISTS hf_secret")
con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

# April + May = historical feature window
feature_data = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_90d,
        SUM(gsc_clicks) AS clicks_90d,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_sum_position) / SUM(gsc_impressions)
            ELSE NULL
        END AS avg_position_90d
    FROM read_parquet([
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/data_0.parquet',
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-05/data_0.parquet'
    ])
    GROUP BY client_hash_id, content_hash_id
""").df()

# June = future outcome window
outcome_data = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_june,
        SUM(gsc_clicks) AS clicks_june
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-06/data_0.parquet'
    )
    GROUP BY client_hash_id, content_hash_id
""").df()

# Combine feature and outcome windows
model_df = feature_data.merge(
    outcome_data,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Same Week-5 population rule
model_df = model_df[
    model_df["impressions_90d"] >= 100
].copy()

# Same Week-5 decline definition
model_df["impressions_change_pct"] = (
    (model_df["impressions_june"] - model_df["impressions_90d"])
    / model_df["impressions_90d"]
) * 100

DECLINE_THRESHOLD = -70

model_df["is_declining_label"] = (
    model_df["impressions_change_pct"] <= DECLINE_THRESHOLD
).astype(int)

# Same Week-5 model features
model_df["ctr_90d"] = np.where(
    model_df["impressions_90d"] > 0,
    model_df["clicks_90d"] / model_df["impressions_90d"],
    0
)

MODEL_FEATURES = [
    "impressions_90d",
    "clicks_90d",
    "avg_position_90d",
    "ctr_90d"
]

print("Model rows:", len(model_df))
print("Clients:", model_df["client_hash_id"].nunique())
print("Declining rate:", f"{model_df['is_declining_label'].mean():.1%}")
print("Features:", MODEL_FEATURES)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Model rows: 136830
Clients: 53
Declining rate: 58.0%
Features: ['impressions_90d', 'clicks_90d', 'avg_position_90d', 'ctr_90d']


###  2B — 5-fold grouped validation by client


In [21]:
def precision_at_k(df, score_column, k):
    return (
        df.sort_values(score_column, ascending=False)
        .head(k)["is_declining_label"]
        .mean()
    )

K_VALUES = [10, 20, 50, 100]

gkf = GroupKFold(n_splits=5)

fold_results = []

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(
        model_df,
        model_df["is_declining_label"],
        groups=model_df["client_hash_id"]
    ),
    start=1
):

    train = model_df.iloc[train_idx].copy()
    test = model_df.iloc[test_idx].copy()

    X_train = train[MODEL_FEATURES]
    y_train = train["is_declining_label"]
    X_test = test[MODEL_FEATURES]

    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE
        ))
    ])

    pipeline.fit(X_train, y_train)

    test["model_score"] = pipeline.predict_proba(X_test)[:, 1]

    result = {
        "fold": fold,
        "train_clients": train["client_hash_id"].nunique(),
        "test_clients": test["client_hash_id"].nunique(),
        "test_declining_rate": test["is_declining_label"].mean()
    }

    for k in K_VALUES:
        result[f"Precision@{k}"] = precision_at_k(
            test,
            "model_score",
            k
        )

    fold_results.append(result)

grouped_results = pd.DataFrame(fold_results)

print("5-Fold Grouped Validation Results")
print(grouped_results.to_string(index=False))

print("\nMean Precision@K:")
for k in K_VALUES:
    print(
        f"Precision@{k}: "
        f"{grouped_results[f'Precision@{k}'].mean():.3f}"
    )

5-Fold Grouped Validation Results
 fold  train_clients  test_clients  test_declining_rate  Precision@10  Precision@20  Precision@50  Precision@100
    1             45             8             0.579207           1.0          0.95          0.94           0.88
    2             43            10             0.467147           0.5          0.70          0.54           0.55
    3             42            11             0.629312           0.9          0.95          0.84           0.80
    4             41            12             0.667714           0.8          0.90          0.88           0.83
    5             41            12             0.555016           1.0          0.95          0.92           0.87

Mean Precision@K:
Precision@10: 0.840
Precision@20: 0.890
Precision@50: 0.824
Precision@100: 0.786


###  2C — Week-5 vs Week-6 validation

In [22]:
week5 = {
    "Precision@10": 0.80,
    "Precision@20": 0.70,
    "Precision@50": 0.68,
    "Precision@100": 0.66
}

comparison = pd.DataFrame({
    "Metric": [f"Precision@{k}" for k in K_VALUES],
    "Week-5 Grouped Holdout": [
        week5[f"Precision@{k}"] for k in K_VALUES
    ],
    "Week-6 5-Fold Grouped Mean": [
        grouped_results[f"Precision@{k}"].mean()
        for k in K_VALUES
    ]
})

comparison["Change"] = (
    comparison["Week-6 5-Fold Grouped Mean"]
    - comparison["Week-5 Grouped Holdout"]
)

print("Before / After Validation Comparison")
print(comparison.to_string(index=False))

print("\nWeek-6 mean test declining rate:",
      f"{grouped_results['test_declining_rate'].mean():.1%}")

print("\nWeek-6 mean test declining rate:",
      f"{grouped_results['test_declining_rate'].mean():.1%}")

print("\nClient overlap checks:")

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(
        model_df,
        model_df["is_declining_label"],
        groups=model_df["client_hash_id"]
    ),
    start=1
):
    train_clients = set(model_df.iloc[train_idx]["client_hash_id"])
    test_clients = set(model_df.iloc[test_idx]["client_hash_id"])

    print(
        f"Fold {fold}:",
        "No overlap" if train_clients.isdisjoint(test_clients)
        else "OVERLAP FOUND"
    )

Before / After Validation Comparison
       Metric  Week-5 Grouped Holdout  Week-6 5-Fold Grouped Mean  Change
 Precision@10                    0.80                       0.840   0.040
 Precision@20                    0.70                       0.890   0.190
 Precision@50                    0.68                       0.824   0.144
Precision@100                    0.66                       0.786   0.126

Week-6 mean test declining rate: 58.0%

Week-6 mean test declining rate: 58.0%

Client overlap checks:
Fold 1: No overlap
Fold 2: No overlap
Fold 3: No overlap
Fold 4: No overlap
Fold 5: No overlap


## 3. Leakage audit

I audited the final four model features against the target and the prediction timeline.

The label is created from the change between historical impressions and June impressions, using a -70% decline threshold. The model features use only the April-May historical window.

I checked for three main leakage risks: label-derived features, future/overlapping outcome information, and identifiers or decision-derived fields being used as model inputs.

The final feature set contains only historical impressions, clicks, average position, and CTR. June outcome columns, the percentage-change calculation, and the target label are not included as model features. Client and content identifiers are used for grouping and joining only.

This supports the conclusion that no direct label-derived or June outcome feature is present in the final model feature list. The audit does not prove that every possible source of bias or leakage has been eliminated, so the result is treated as an observed validation check rather than proof of a completely leakage-free system.

In [23]:
# 1. Check the final feature set against known target/future columns
label_and_future_columns = [
    "impressions_june",
    "clicks_june",
    "impressions_change_pct",
    "is_declining_label"
]

feature_set = set(MODEL_FEATURES)

direct_leakage = feature_set.intersection(
    label_and_future_columns
)

print("Final model features:")
print(MODEL_FEATURES)

print("\nKnown label/future columns:")
print(label_and_future_columns)

print("\nDirect leakage detected:",
      direct_leakage if direct_leakage else "None")


# 2. Check that identifiers are not model features

identifier_columns = {
    "client_hash_id",
    "content_hash_id"
}

identifier_leakage = feature_set.intersection(
    identifier_columns
)

print("\nIdentifier leakage:",
      identifier_leakage if identifier_leakage else "None")


# 3. Check the feature timeline

print("\nTimeline check:")
print("Features: April + May historical data")
print("Outcome: June data")
print("Label: June vs historical impressions")
print("Result: Features occur before the June outcome window.")

print("\nFeature source check:")
print("impressions_90d, clicks_90d, avg_position_90d, and ctr_90d")
print("are calculated from the April-May historical feature window.")
print("June outcome variables are used only to construct the label.")

# 4. Check that the target exists but is not in X

X_columns = set(MODEL_FEATURES)
target_column = "is_declining_label"

print("\nTarget included as a feature:",
      target_column in X_columns)


# 5. Final verdict

if (
    not direct_leakage
    and not identifier_leakage
    and target_column not in X_columns
):
    print("\nLeakage audit verdict: No direct leakage found in the final feature set.")
else:
    print("\nLeakage audit verdict: Review required.")

Final model features:
['impressions_90d', 'clicks_90d', 'avg_position_90d', 'ctr_90d']

Known label/future columns:
['impressions_june', 'clicks_june', 'impressions_change_pct', 'is_declining_label']

Direct leakage detected: None

Identifier leakage: None

Timeline check:
Features: April + May historical data
Outcome: June data
Label: June vs historical impressions
Result: Features occur before the June outcome window.

Feature source check:
impressions_90d, clicks_90d, avg_position_90d, and ctr_90d
are calculated from the April-May historical feature window.
June outcome variables are used only to construct the label.

Target included as a feature: False

Leakage audit verdict: No direct leakage found in the final feature set.


## 4. Claim rewrite

### Original claim

"Logistic Regression performed better than the Week-4 baseline at every tested K."

### Safer claim

"On the evaluated grouped test data, Logistic Regression showed higher measured Precision@K than the Week-4 rule-based baseline at K=10, 20, 50, and 100. Across the Week-6 5-fold grouped validation, the model measured mean Precision@10 of 0.84, Precision@20 of 0.89, Precision@50 of 0.82, and Precision@100 of 0.79.

These results provide a directional ranking signal that may support content-refresh opportunity prioritization. They do not establish that refreshing a high-scoring page will cause traffic improvement or guarantee a better outcome."

The model should therefore be treated as decision-support for prioritizing refresh opportunities, rather than as an automatic refresh decision or a guarantee of future performance.

In [24]:
claim_metrics = {
    "Precision@10": 0.840,
    "Precision@20": 0.890,
    "Precision@50": 0.824,
    "Precision@100": 0.786
}

print("Measured Week-6 5-fold grouped mean:")
for metric, value in claim_metrics.items():
    print(f"{metric}: {value:.3f}")

print("\nInterpretation:")
print("Observed and measured ranking performance on the evaluated grouped folds.")
print("Use: directional decision-support for content-refresh prioritization.")
print("Not established: causal traffic improvement or guaranteed refresh outcomes.")

Measured Week-6 5-fold grouped mean:
Precision@10: 0.840
Precision@20: 0.890
Precision@50: 0.824
Precision@100: 0.786

Interpretation:
Observed and measured ranking performance on the evaluated grouped folds.
Use: directional decision-support for content-refresh prioritization.
Not established: causal traffic improvement or guaranteed refresh outcomes.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.